In [1]:
import numpy as np
from scipy.optimize import minimize
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

Soccer Shot Selection Optimization problem: A soccer team wants to choose how to distribute its shots during a game in order to avoid wasting scoring opportunities. Some shots (like close-range shots) are more effective, while others (like long shots) are less efficient. The team must balance different shot types while staying within realistic limits on total shots and game situations.

\begin{eqnarray*}
\min f(x_1,x_2,x_3,x_4,x_5) = 0.55x_1^2 + 0.82x_2^2 + 0.95x_3^2 + 0.90x_4^2 + 0.75x_5^2 + 0.20x_3x_4 - 0.15x_1x_5 \\
\begin{array}{c} 
\mbox{subject to} \hspace{5in} \\
x_1 + x_2 + x_3 + x_4 + x_5 = 15 \hspace{2in} \\
x_3 \le 4 \hspace{2in} \\
x_1 + x_2 \ge 8 \hspace{2in} \\
x_5 \le 5 \hspace {2in} \\
x_i \ge 0 \mbox{ for all i} \hspace{2in} \\
\end{array} \\
\end{eqnarray*}

\begin{align*}
x_1 &= \text{number of shots taken from inside the 6-yard box (close range)} \\
x_2 &= \text{number of shots taken from inside the penalty box, but outside the 6-yard box (medium range)} \\
x_3 &= \text{number of shots taken from outside the penalty box (long range)} \\
x_4 &= \text{number of shots taken with the head} \\
x_5 &= \text{number of shots taken during counterattacks}
\end{align*}

* The objective function represents a measure of inefficiency in shot selection. Larger values indicate more wasted opportunities due to poor shot quality or overuse of certain shot types, while smaller values indicate a more efficient offensive strategy.
* The first constraint ensures the team takes a realistic number of shots
* The second constraint limits inefficient shot types
* The third constraint ensures enough high-quality scoring attempts
* The fourth constraint reflects limited real-game opportunities
* The fifth constraint ensures valid shot counts

Toy Version: We consider only two types of shots: high-quality close shots and low-quality long shots. The goal is to find the best balance between these two types while keeping the total number of shots fixed.
\begin{eqnarray*}
\min f(x_1,x_3) = 0.55x_1^2 + 0.95x_3^2 - 0.10x_1x_3 \\
\begin{array}{c} 
\mbox{subject to} \hspace{2in} \\
x_1 + x_3 = 10 \\
x_1,x_3 \ge 0 \\
\end{array} \\
\end{eqnarray*}


\begin{align*}
x_1 &= \text{number of shots taken from inside the 6-yard box (close range)} \\
x_3 &= \text{number of shots taken from outside the penalty box (long range)} \\
\end{align*}

* The objective function represents the inefficiency of the shot distribution, where taking too many of one type increases inefficiency, and mixing shot types slightly improves performance
* The first constraint fixes the total number of shots
* The second constraint ensures valid shot counts

Convexity: 
$$
\frac{\partial f}{\partial x_1}
= 1.10x_1 - 0.10x_3
$$

$$
\frac{\partial f}{\partial x_3}
= 1.90x_3 - 0.10x_1
$$

$$
\frac{\partial^2 f}{\partial x_1^2}
= 1.10
$$

$$
\frac{\partial^2 f}{\partial x_3^2}
= 1.90
$$

$$
\frac{\partial^2 f}{\partial x_1 \partial x_3}
= -0.10
$$

$$
H =
\begin{bmatrix}
1.10 & -0.10 \\
-0.10 & 1.90
\end{bmatrix}
$$

$$
\det([1.1]) = 1.1 > 0
$$

$$
\det([1.90]) = 1.90 > 0
$$

$$
\det(H)
= (1.10)(1.90) - (-0.10)^2
= 2.08 > 0
$$
Objective function is convex. Constraints are linear so feasible region is convex.

In [3]:
# objective function
def f(x):
    x1, x3 = x
    return 0.55*x1**2 + 0.95*x3**2 - 0.10*x1*x3

# gradient of the objective function
def grad_f(x):
    x1, x3 = x
    return np.array([
        1.1*x1 - 0.10*x3,
        1.9*x3 - 0.10*x1
    ])

# SLSQP assumes that inequality constraints are of the form g(x) >= 0
constraints = [
    {
        "type": "eq",
        "fun": lambda x: x[0] + x[1] - 10,
        "jac": lambda x: np.array([1, 1])
    }
]

bounds = [(0, None)]

In [4]:
initial_points = [np.array([5, 5]), np.array([8, 2]), np.array([2, 8]), 
                  np.array([3,7]), np.array([6,4])]

results = []

def run_slsqp(x0):
    res = minimize(
        f, x0,
        method="SLSQP",
        jac=grad_f,
        bounds=bounds,
        constraints=constraints,
        options={"ftol": 1e-12, "disp": True}
    )

    print("Status:", res.message)
    print("Solution:", res.x)
    print("Objective:", res.fun)

    return res

In [5]:
res = run_slsqp(np.array([5, 5]))

Optimization terminated successfully    (Exit mode 0)
            Current function value: 32.5
            Iterations: 3
            Function evaluations: 3
            Gradient evaluations: 3
Status: Optimization terminated successfully
Solution: [6.25 3.75]
Objective: 32.5


In [6]:
res = run_slsqp(np.array([8, 2]))

Optimization terminated successfully    (Exit mode 0)
            Current function value: 32.5
            Iterations: 3
            Function evaluations: 3
            Gradient evaluations: 3
Status: Optimization terminated successfully
Solution: [6.25 3.75]
Objective: 32.5


In [7]:
res = run_slsqp(np.array([2, 8]))

Optimization terminated successfully    (Exit mode 0)
            Current function value: 32.5
            Iterations: 3
            Function evaluations: 3
            Gradient evaluations: 3
Status: Optimization terminated successfully
Solution: [6.25 3.75]
Objective: 32.5


In [8]:
res = run_slsqp(np.array([3, 7]))

Optimization terminated successfully    (Exit mode 0)
            Current function value: 32.50000000000001
            Iterations: 3
            Function evaluations: 3
            Gradient evaluations: 3
Status: Optimization terminated successfully
Solution: [6.25 3.75]
Objective: 32.50000000000001


In [9]:
res = run_slsqp(np.array([6, 4]))

Optimization terminated successfully    (Exit mode 0)
            Current function value: 32.5
            Iterations: 3
            Function evaluations: 3
            Gradient evaluations: 3
Status: Optimization terminated successfully
Solution: [6.25 3.75]
Objective: 32.5


This solution is reasonable because:
* it balances close and long shots rather than choosing extremes
* it reflects diminishing returns from overusing any one shot type
* it is consistent across multiple starting points

In [10]:
from scipy.optimize import minimize, NonlinearConstraint

# We use the same objective defined by f(x) above and we use the same
# constraint structure as the one for differential evolution.
def c1(x):
    return x[0] + x[1]

nlc1 = NonlinearConstraint(c1, 10, 10)

x0 = np.array([5, 5])

res_cobyqa = minimize(
    f,
    x0,
    method="COBYQA",
    constraints=[nlc1],
    options={"maxiter": 2000, "feasibility_tol": 1e-9}
)

print("COBYQA solution:", res_cobyqa.x)
print("COBYQA objective:", res_cobyqa.fun)
print("message:", res_cobyqa.message)

COBYQA solution: [6.25 3.75]
COBYQA objective: 32.500000000000014
message: The lower bound for the trust-region radius has been reached


In [13]:
# --------------------------------------------------
# Grid for visualization
# --------------------------------------------------
run_slsqp(np.array([5, 5]))
x1_star, x3_star = res.x
f_star = res.fun

x1_vals = np.linspace(0, 10, 120)
x3_vals = np.linspace(0, 10, 120)
x1, x3 = np.meshgrid(x1_vals, x3_vals)

Z = 0.55*x1**2 + 0.95*x3**2 - 0.10*x1*x3

# Feasible region mask
feasible = np.isclose(x1 + x3, 10.0, atol=0.15)

# Objective over feasible region only
Z_feas = np.where(feasible, Z, np.nan)

# --------------------------------------------------
# 3D surface plot
# --------------------------------------------------
surface_all = go.Surface(
    x=x1,
    y=x3,
    z=Z,
    opacity=0.72,
    colorscale="Viridis",
    name="Objective surface",
    showscale=True
)

surface_feasible = go.Surface(
    x=x1,
    y=x3,
    z=Z_feas,
    opacity=0.95,
    colorscale="Plasma",
    name="Feasible part",
    showscale=False
)

opt_point_3d = go.Scatter3d(
    x=[x1_star],
    y=[x3_star],
    z=[f_star],
    mode="markers+text",
    text=["SLSQP solution"],
    textposition="top center",
    marker=dict(size=6, color="red"),
    name="SLSQP solution"
)

fig3d = go.Figure(data=[surface_all, surface_feasible, opt_point_3d])

fig3d.update_layout(
    title="Toy Soccer Optimization: Objective Surface and Feasible Region",
    scene=dict(
        xaxis_title="x1 (close shots)",
        yaxis_title="x3 (long shots)",
        zaxis_title="f(x1, x3)"
    ),
    width=950,
    height=750
)

fig3d.show()

Optimization terminated successfully    (Exit mode 0)
            Current function value: 32.5
            Iterations: 3
            Function evaluations: 3
            Gradient evaluations: 3
Status: Optimization terminated successfully
Solution: [6.25 3.75]
Objective: 32.5


In [12]:
# --------------------------------------------------
# 2D contour plot + feasible region
# --------------------------------------------------
contours = go.Contour(
    x=x1_vals,
    y=x3_vals,
    z=Z,
    colorscale="Viridis",
    contours=dict(showlabels=True),
    opacity=0.85,
    name="Objective contours"
)

# Shade feasible region
feasible_region = go.Contour(
    x=x1_vals,
    y=x3_vals,
    z=feasible.astype(int),
    showscale=False,
    contours=dict(start=0.5, end=1.5, size=1),
    opacity=0.28,
    colorscale=[[0, "lightgreen"], [1, "lightgreen"]],
    name="Feasible region"
)

x_line = np.linspace(0, 10, 500)
y_line = 10 - x_line

area_boundary = go.Scatter(
    x=x_line,
    y=y_line,
    mode="lines",
    line=dict(color="blue", width=2),
    name="x1 + x3 = 10"
)

# Optimal point
opt_point_2d = go.Scatter(
    x=[x1_star],
    y=[x3_star],
    mode="markers+text",
    text=["SLSQP solution"],
    textposition="top center",
    marker=dict(size=10, color="red"),
    name="SLSQP solution"
)

fig2d = go.Figure(data=[
    contours,
    feasible_region,
    area_boundary,
    opt_point_2d
])

fig2d.update_layout(
    title="Toy Soccer Optimization: Feasible Region and Objective Contours",
    xaxis_title="x1 (close shots)",
    yaxis_title="x3 (long shots)",
    width=900,
    height=750
)

fig2d.show()